# Traditional Score Extraction

General-purpose notebook for extracting traditional evaluation metrics from model responses.

**Required DataFrame columns:**
- `response` — model-generated answer
- `gt` — ground truth label

**Metrics computed:** BLEU, ROUGE-L, ROUGE-Lsum, BERT Precision/Recall/F1, METEOR

## Environment Setup

In [ ]:
import sys
import warnings

if not sys.warnoptions:
    warnings.simplefilter("ignore")

In [ ]:
!pip install -q evaluate bert-score rouge_score absl-py

## Imports

In [ ]:
import pandas as pd
import json
import os

import evaluate
from bert_score import score as bert_score
from tqdm import tqdm

## Load Evaluation Metrics

In [ ]:
bleu_metric = evaluate.load("bleu")
rouge_metric = evaluate.load("rouge")
meteor_metric = evaluate.load("meteor")

## Metric Computation Functions

In [ ]:
def compute_bleu(predictions, references, max_order=2):
    if not predictions or not references:
        return {"BLEU": 0.0}
    score = bleu_metric.compute(
        predictions=predictions,
        references=[[ref] for ref in references],
        max_order=max_order,
        smooth=True
    )["bleu"]
    return {"BLEU": score}


def compute_rouge_l_and_sum(predictions, references):
    if not predictions or not references:
        return {"RougeL": 0.0, "RougeLsum": 0.0}
    results = rouge_metric.compute(
        predictions=predictions,
        references=references,
        use_stemmer=True
    )
    return {
        "RougeL": results["rougeL"],
        "RougeLsum": results["rougeLsum"]
    }


def compute_bertscore(predictions, references, lang="en"):
    if not predictions or not references:
        return {"BERT Precision": 0.0, "BERT Recall": 0.0, "BERT F1": 0.0}
    P, R, F1 = bert_score(predictions, references, lang=lang, verbose=False)
    return {
        "BERT Precision": P.mean().item(),
        "BERT Recall": R.mean().item(),
        "BERT F1": F1.mean().item()
    }


def compute_meteor(predictions, references):
    if not predictions or not references:
        return {"METEOR": 0.0}
    score = meteor_metric.compute(
        predictions=predictions,
        references=references
    )["meteor"]
    return {"METEOR": score}

## Zero-Shot Score Generation

Computes all metrics row-by-row using `response` and `gt` columns.

In [ ]:
def zero_shot_generate_score(df):
    """Compute all evaluation metrics for zero-shot responses.

    Args:
        df: DataFrame with 'response' and 'gt' columns.

    Returns:
        DataFrame with zero-shot score columns appended.
    """
    metrics = {
        "zero-shot-BLEU": [], "zero-shot-RougeL": [], "zero-shot-RougeLsum": [],
        "zero-shot-BERT-Precision": [], "zero-shot-BERT Recall": [], "zero-shot-BERT F1": [],
        "zero-shot-METEOR": []
    }

    for _, row in tqdm(df.iterrows(), total=len(df), desc="Scoring Zero-Shot"):
        ref = str(row.get("gt", ""))
        pred = str(row.get("response", ""))

        try:
            bleu = compute_bleu([pred], [ref])
            rouge = compute_rouge_l_and_sum([pred], [ref])
            bert = compute_bertscore([pred], [ref])
            meteor = compute_meteor([pred], [ref])

            metrics["zero-shot-BLEU"].append(bleu["BLEU"])
            metrics["zero-shot-RougeL"].append(rouge["RougeL"])
            metrics["zero-shot-RougeLsum"].append(rouge["RougeLsum"])
            metrics["zero-shot-BERT-Precision"].append(bert["BERT Precision"])
            metrics["zero-shot-BERT Recall"].append(bert["BERT Recall"])
            metrics["zero-shot-BERT F1"].append(bert["BERT F1"])
            metrics["zero-shot-METEOR"].append(meteor["METEOR"])

        except Exception as e:
            print(f"Error processing row: {e}")
            for key in metrics:
                metrics[key].append(None)

    for key, values in metrics.items():
        df[key] = values

    return df

## CoT Score Generation

Computes all metrics row-by-row using `response` and `gt` columns for Chain-of-Thought responses.

In [ ]:
def cot_generate_score(df):
    """Compute all evaluation metrics for CoT responses.

    Args:
        df: DataFrame with 'response' and 'gt' columns.

    Returns:
        DataFrame with CoT score columns appended.
    """
    metrics = {
        "cot-BLEU": [], "cot-RougeL": [], "cot-RougeLsum": [],
        "cot-BERT-Precision": [], "cot-BERT-Recall": [], "cot-BERT-F1": [],
        "cot-METEOR": []
    }

    for _, row in tqdm(df.iterrows(), total=len(df), desc="Scoring CoT"):
        ref = str(row.get("gt", ""))
        pred = str(row.get("response", ""))

        try:
            bleu = compute_bleu([pred], [ref])
            rouge = compute_rouge_l_and_sum([pred], [ref])
            bert = compute_bertscore([pred], [ref])
            meteor = compute_meteor([pred], [ref])

            metrics["cot-BLEU"].append(bleu["BLEU"])
            metrics["cot-RougeL"].append(rouge["RougeL"])
            metrics["cot-RougeLsum"].append(rouge["RougeLsum"])
            metrics["cot-BERT-Precision"].append(bert["BERT Precision"])
            metrics["cot-BERT-Recall"].append(bert["BERT Recall"])
            metrics["cot-BERT-F1"].append(bert["BERT F1"])
            metrics["cot-METEOR"].append(meteor["METEOR"])

        except Exception as e:
            print(f"Error processing row: {e}")
            for key in metrics:
                metrics[key].append(None)

    for key, values in metrics.items():
        df[key] = values

    return df

## Load Response Data

Load ground truth from cloud/device test sets, then load the response file to score.
The `gt` column comes from `selected_answer` in the test sets.

### Load Ground Truth

In [ ]:
# --- Cloud Ground Truth ---
# cloud_test = pd.read_excel("../../data/splits/cloud/cloud_test.xlsx")
# cloud_gt = cloud_test[["selected_answer"]].rename(columns={"selected_answer": "gt"})

# --- Device Ground Truth ---
# device_test = pd.read_excel("../../data/splits/device/device_test.xlsx")
# device_gt = device_test[["selected_answer"]].rename(columns={"selected_answer": "gt"})

### Load Response Files

Uncomment the response file you want to score. Merge with the corresponding ground truth.

### Baselines — LLM + OCR

In [ ]:
# --- Cloud ---
# df = pd.read_excel("../../outputs/responses/baselines/llm_ocr/cloud/llm_ocr_response_cloud_deepseek_r1_distill_qwen_7b_zero_shot.xlsx")
# df = pd.read_excel("../../outputs/responses/baselines/llm_ocr/cloud/llm_ocr_response_cloud_deepseek_r1_distill_qwen_7b_cot.xlsx")
# df = pd.read_excel("../../outputs/responses/baselines/llm_ocr/cloud/llm_ocr_response_cloud_llama_3_1_8b_instruct_zero_shot.xlsx")
# df = pd.read_excel("../../outputs/responses/baselines/llm_ocr/cloud/llm_ocr_response_cloud_llama_3_1_8b_instruct_cot.xlsx")
# df = pd.read_excel("../../outputs/responses/baselines/llm_ocr/cloud/llm_ocr_response_cloud_mistral_8b_instruct_2410_zero_shot.xlsx")
# df = pd.read_excel("../../outputs/responses/baselines/llm_ocr/cloud/llm_ocr_response_cloud_mistral_8b_instruct_2410_cot.xlsx")
# df = pd.read_excel("../../outputs/responses/baselines/llm_ocr/cloud/llm_ocr_response_cloud_qwen_7b_zero_shot.xlsx")
# df = pd.read_excel("../../outputs/responses/baselines/llm_ocr/cloud/llm_ocr_response_cloud_qwen_7b_cot.xlsx")

# --- Device ---
# df = pd.read_excel("../../outputs/responses/baselines/llm_ocr/device/llm_ocr_response_device_deepseek_r1_distill_qwen_7b_zero_shot.xlsx")
# df = pd.read_excel("../../outputs/responses/baselines/llm_ocr/device/llm_ocr_response_device_deepseek_r1_distill_qwen_7b_cot.xlsx")
# df = pd.read_excel("../../outputs/responses/baselines/llm_ocr/device/llm_ocr_response_device_llama_3_1_8b_instruct_zero_shot.xlsx")
# df = pd.read_excel("../../outputs/responses/baselines/llm_ocr/device/llm_ocr_response_device_llama_3_1_8b_instruct_cot.xlsx")
# df = pd.read_excel("../../outputs/responses/baselines/llm_ocr/device/llm_ocr_response_device_mistral_8b_instruct_2410_zero_shot.xlsx")
# df = pd.read_excel("../../outputs/responses/baselines/llm_ocr/device/llm_ocr_response_device_mistral_8b_instruct_2410_cot.xlsx")
# df = pd.read_excel("../../outputs/responses/baselines/llm_ocr/device/llm_ocr_response_device_qwen_7b_zero_shot.xlsx")
# df = pd.read_excel("../../outputs/responses/baselines/llm_ocr/device/llm_ocr_response_device_qwen_7b_cot.xlsx")

### Baselines — LLM + Text

In [ ]:
# --- Cloud ---
# df = pd.read_excel("../../outputs/responses/baselines/llm_text/cloud/llm_text_response_cloud_deepseek_r1_distill_qwen_7b_zero_shot.xlsx")
# df = pd.read_excel("../../outputs/responses/baselines/llm_text/cloud/llm_text_response_cloud_deepseek_r1_distill_qwen_7b_cot.xlsx")
# df = pd.read_excel("../../outputs/responses/baselines/llm_text/cloud/llm_text_response_cloud_llama_3_1_8b_instruct_zero_shot.xlsx")
# df = pd.read_excel("../../outputs/responses/baselines/llm_text/cloud/llm_text_response_cloud_llama_3_1_8b_instruct_cot.xlsx")
# df = pd.read_excel("../../outputs/responses/baselines/llm_text/cloud/llm_text_response_cloud_mistral_8b_instruct_2410_zero_shot.xlsx")
# df = pd.read_excel("../../outputs/responses/baselines/llm_text/cloud/llm_text_response_cloud_mistral_8b_instruct_2410_cot.xlsx")
# df = pd.read_excel("../../outputs/responses/baselines/llm_text/cloud/llm_text_response_cloud_qwen_7b_zero_shot.xlsx")
# df = pd.read_excel("../../outputs/responses/baselines/llm_text/cloud/llm_text_response_cloud_qwen_7b_cot.xlsx")

# --- Device ---
# df = pd.read_excel("../../outputs/responses/baselines/llm_text/device/llm_text_response_device_deepseek_r1_distill_qwen_7b_zero_shot.xlsx")
# df = pd.read_excel("../../outputs/responses/baselines/llm_text/device/llm_text_response_device_deepseek_r1_distill_qwen_7b_cot.xlsx")
# df = pd.read_excel("../../outputs/responses/baselines/llm_text/device/llm_text_response_device_llama_3_1_8b_instruct_zero_shot.xlsx")
# df = pd.read_excel("../../outputs/responses/baselines/llm_text/device/llm_text_response_device_llama_3_1_8b_instruct_cot.xlsx")
# df = pd.read_excel("../../outputs/responses/baselines/llm_text/device/llm_text_response_device_mistral_8b_instruct_2410_zero_shot.xlsx")
# df = pd.read_excel("../../outputs/responses/baselines/llm_text/device/llm_text_response_device_mistral_8b_instruct_2410_cot.xlsx")
# df = pd.read_excel("../../outputs/responses/baselines/llm_text/device/llm_text_response_device_qwen_7b_zero_shot.xlsx")
# df = pd.read_excel("../../outputs/responses/baselines/llm_text/device/llm_text_response_device_qwen_7b_cot.xlsx")

### Baselines — VLM

In [ ]:
# --- Cloud ---
# df = pd.read_excel("../../outputs/responses/baselines/vlm/cloud/vlm_response_cloud_llama_3_1_8b_instruct_zero_shot.xlsx")
# df = pd.read_excel("../../outputs/responses/baselines/vlm/cloud/vlm_response_cloud_llama_3_1_8b_instruct_cot.xlsx")
# df = pd.read_excel("../../outputs/responses/baselines/vlm/cloud/vlm_response_cloud_llava1_5_vl_7b_zero_shot.xlsx")
# df = pd.read_excel("../../outputs/responses/baselines/vlm/cloud/vlm_response_cloud_llava1_5_vl_7b_cot.xlsx")
# df = pd.read_excel("../../outputs/responses/baselines/vlm/cloud/vlm_response_cloud_qwen_7b_zero_shot.xlsx")
# df = pd.read_excel("../../outputs/responses/baselines/vlm/cloud/vlm_response_cloud_qwen_7b_cot.xlsx")

# --- Device ---
# df = pd.read_excel("../../outputs/responses/baselines/vlm/device/vlm_response_device_llama_3_1_8b_instruct_zero_shot.xlsx")
# df = pd.read_excel("../../outputs/responses/baselines/vlm/device/vlm_response_device_llama_3_1_8b_instruct_cot.xlsx")
# df = pd.read_excel("../../outputs/responses/baselines/vlm/device/vlm_response_device_llava1_5_vl_7b_zero_shot.xlsx")
# df = pd.read_excel("../../outputs/responses/baselines/vlm/device/vlm_response_device_llava1_5_vl_7b_cot.xlsx")
# df = pd.read_excel("../../outputs/responses/baselines/vlm/device/vlm_response_device_qwen_7b_zero_shot.xlsx")
# df = pd.read_excel("../../outputs/responses/baselines/vlm/device/vlm_response_device_qwen_7b_cot.xlsx")

### Fine-tuned — SFT

In [ ]:
# --- Cloud ---
# df = pd.read_excel("../../outputs/responses/sft/cloud/sft_llm_ocr_train_cloud_llama_zero_shot.xlsx")

# --- Device ---
# df = pd.read_excel("../../outputs/responses/sft/device/sft_llm_ocr_train_device_llama_zero_shot.xlsx")

### Fine-tuned — DPO

In [ ]:
# --- Cloud ---
# df = pd.read_excel("../../outputs/responses/dpo/cloud/dpo_llm_ocr_train_cloud_llama_zero_shot.xlsx")

# --- Device ---
# df = pd.read_excel("../../outputs/responses/dpo/device/dpo_llm_ocr_train_device_llama_zero_shot.xlsx")

## Verify & Merge Data

In [ ]:
# Merge response with ground truth (use cloud_gt or device_gt as appropriate)
# df["gt"] = cloud_gt["gt"].values
# df["gt"] = device_gt["gt"].values

# assert "response" in df.columns, "Missing 'response' column"
# assert "gt" in df.columns, "Missing 'gt' column"
# print(f"Loaded {len(df)} rows")
# df[["response", "gt"]].head(3)

## Run Score Extraction

In [ ]:
# For zero-shot responses:
# df_scored = zero_shot_generate_score(df)

# For CoT responses:
# df_scored = cot_generate_score(df)

## View Results

In [ ]:
# Zero-shot score columns
zero_shot_score_cols = [
    "zero-shot-BLEU", "zero-shot-RougeL", "zero-shot-RougeLsum",
    "zero-shot-BERT-Precision", "zero-shot-BERT Recall", "zero-shot-BERT F1",
    "zero-shot-METEOR"
]

# CoT score columns
cot_score_cols = [
    "cot-BLEU", "cot-RougeL", "cot-RougeLsum",
    "cot-BERT-Precision", "cot-BERT-Recall", "cot-BERT-F1",
    "cot-METEOR"
]

In [ ]:
# Average scores (uncomment the relevant one)
# df_scored[zero_shot_score_cols].mean()
# df_scored[cot_score_cols].mean()

In [ ]:
# Detailed statistics
# df_scored[zero_shot_score_cols].describe()
# df_scored[cot_score_cols].describe()

## Save Scored Results

### Baselines — LLM + OCR Scores

In [ ]:
# --- Cloud ---
# df_scored.to_excel("../../outputs/scores/baselines/llm_ocr/cloud/llm_ocr_score_cloud_deepseek_r1_distill_qwen_7b_zero_shot.xlsx", index=False)
# df_scored.to_excel("../../outputs/scores/baselines/llm_ocr/cloud/llm_ocr_score_cloud_deepseek_r1_distill_qwen_7b_cot.xlsx", index=False)
# df_scored.to_excel("../../outputs/scores/baselines/llm_ocr/cloud/llm_ocr_score_cloud_llama_3_1_8b_instruct_zero_shot.xlsx", index=False)
# df_scored.to_excel("../../outputs/scores/baselines/llm_ocr/cloud/llm_ocr_score_cloud_llama_3_1_8b_instruct_cot.xlsx", index=False)
# df_scored.to_excel("../../outputs/scores/baselines/llm_ocr/cloud/llm_ocr_score_cloud_mistral_8b_instruct_2410_zero_shot.xlsx", index=False)
# df_scored.to_excel("../../outputs/scores/baselines/llm_ocr/cloud/llm_ocr_score_cloud_mistral_8b_instruct_2410_cot.xlsx", index=False)
# df_scored.to_excel("../../outputs/scores/baselines/llm_ocr/cloud/llm_ocr_score_cloud_qwen_7b_zero_shot.xlsx", index=False)
# df_scored.to_excel("../../outputs/scores/baselines/llm_ocr/cloud/llm_ocr_score_cloud_qwen_7b_cot.xlsx", index=False)

# --- Device ---
# df_scored.to_excel("../../outputs/scores/baselines/llm_ocr/device/llm_ocr_score_device_deepseek_r1_distill_qwen_7b_zero_shot.xlsx", index=False)
# df_scored.to_excel("../../outputs/scores/baselines/llm_ocr/device/llm_ocr_score_device_deepseek_r1_distill_qwen_7b_cot.xlsx", index=False)
# df_scored.to_excel("../../outputs/scores/baselines/llm_ocr/device/llm_ocr_score_device_llama_3_1_8b_instruct_zero_shot.xlsx", index=False)
# df_scored.to_excel("../../outputs/scores/baselines/llm_ocr/device/llm_ocr_score_device_llama_3_1_8b_instruct_cot.xlsx", index=False)
# df_scored.to_excel("../../outputs/scores/baselines/llm_ocr/device/llm_ocr_score_device_mistral_8b_instruct_2410_zero_shot.xlsx", index=False)
# df_scored.to_excel("../../outputs/scores/baselines/llm_ocr/device/llm_ocr_score_device_mistral_8b_instruct_2410_cot.xlsx", index=False)
# df_scored.to_excel("../../outputs/scores/baselines/llm_ocr/device/llm_ocr_score_device_qwen_7b_zero_shot.xlsx", index=False)
# df_scored.to_excel("../../outputs/scores/baselines/llm_ocr/device/llm_ocr_score_device_qwen_7b_cot.xlsx", index=False)

### Baselines — LLM + Text Scores

In [ ]:
# --- Cloud ---
# df_scored.to_excel("../../outputs/scores/baselines/llm_text/cloud/llm_text_score_cloud_deepseek_r1_distill_qwen_7b_zero_shot.xlsx", index=False)
# df_scored.to_excel("../../outputs/scores/baselines/llm_text/cloud/llm_text_score_cloud_deepseek_r1_distill_qwen_7b_cot.xlsx", index=False)
# df_scored.to_excel("../../outputs/scores/baselines/llm_text/cloud/llm_text_score_cloud_llama_3_1_8b_instruct_zero_shot.xlsx", index=False)
# df_scored.to_excel("../../outputs/scores/baselines/llm_text/cloud/llm_text_score_cloud_llama_3_1_8b_instruct_cot.xlsx", index=False)
# df_scored.to_excel("../../outputs/scores/baselines/llm_text/cloud/llm_text_score_cloud_mistral_8b_instruct_2410_zero_shot.xlsx", index=False)
# df_scored.to_excel("../../outputs/scores/baselines/llm_text/cloud/llm_text_score_cloud_mistral_8b_instruct_2410_cot.xlsx", index=False)
# df_scored.to_excel("../../outputs/scores/baselines/llm_text/cloud/llm_text_score_cloud_qwen_7b_zero_shot.xlsx", index=False)
# df_scored.to_excel("../../outputs/scores/baselines/llm_text/cloud/llm_text_score_cloud_qwen_7b_cot.xlsx", index=False)

# --- Device ---
# df_scored.to_excel("../../outputs/scores/baselines/llm_text/device/llm_text_score_device_deepseek_r1_distill_qwen_7b_zero_shot.xlsx", index=False)
# df_scored.to_excel("../../outputs/scores/baselines/llm_text/device/llm_text_score_device_deepseek_r1_distill_qwen_7b_cot.xlsx", index=False)
# df_scored.to_excel("../../outputs/scores/baselines/llm_text/device/llm_text_score_device_llama_3_1_8b_instruct_zero_shot.xlsx", index=False)
# df_scored.to_excel("../../outputs/scores/baselines/llm_text/device/llm_text_score_device_llama_3_1_8b_instruct_cot.xlsx", index=False)
# df_scored.to_excel("../../outputs/scores/baselines/llm_text/device/llm_text_score_device_mistral_8b_instruct_2410_zero_shot.xlsx", index=False)
# df_scored.to_excel("../../outputs/scores/baselines/llm_text/device/llm_text_score_device_mistral_8b_instruct_2410_cot.xlsx", index=False)
# df_scored.to_excel("../../outputs/scores/baselines/llm_text/device/llm_text_score_device_qwen_7b_zero_shot.xlsx", index=False)
# df_scored.to_excel("../../outputs/scores/baselines/llm_text/device/llm_text_score_device_qwen_7b_cot.xlsx", index=False)

### Baselines — VLM Scores

In [ ]:
# --- Cloud ---
# df_scored.to_excel("../../outputs/scores/baselines/vlm/cloud/vlm_score_cloud_llama_3_1_8b_instruct_zero_shot.xlsx", index=False)
# df_scored.to_excel("../../outputs/scores/baselines/vlm/cloud/vlm_score_cloud_llama_3_1_8b_instruct_cot.xlsx", index=False)
# df_scored.to_excel("../../outputs/scores/baselines/vlm/cloud/vlm_score_cloud_llava1_5_vl_7b_zero_shot.xlsx", index=False)
# df_scored.to_excel("../../outputs/scores/baselines/vlm/cloud/vlm_score_cloud_llava1_5_vl_7b_cot.xlsx", index=False)
# df_scored.to_excel("../../outputs/scores/baselines/vlm/cloud/vlm_score_cloud_qwen_7b_zero_shot.xlsx", index=False)
# df_scored.to_excel("../../outputs/scores/baselines/vlm/cloud/vlm_score_cloud_qwen_7b_cot.xlsx", index=False)

# --- Device ---
# df_scored.to_excel("../../outputs/scores/baselines/vlm/device/vlm_score_device_llama_3_1_8b_instruct_zero_shot.xlsx", index=False)
# df_scored.to_excel("../../outputs/scores/baselines/vlm/device/vlm_score_device_llama_3_1_8b_instruct_cot.xlsx", index=False)
# df_scored.to_excel("../../outputs/scores/baselines/vlm/device/vlm_score_device_llava1_5_vl_7b_zero_shot.xlsx", index=False)
# df_scored.to_excel("../../outputs/scores/baselines/vlm/device/vlm_score_device_llava1_5_vl_7b_cot.xlsx", index=False)
# df_scored.to_excel("../../outputs/scores/baselines/vlm/device/vlm_score_device_qwen_7b_zero_shot.xlsx", index=False)
# df_scored.to_excel("../../outputs/scores/baselines/vlm/device/vlm_score_device_qwen_7b_cot.xlsx", index=False)

### Fine-tuned — SFT Scores

In [ ]:
# --- Cloud ---
# df_scored.to_excel("../../outputs/scores/sft/cloud/sft_llama3_instruct_cloud_zero_shot_score.xlsx", index=False)

# --- Device ---
# df_scored.to_excel("../../outputs/scores/sft/device/sft_llama3_instruct_device_zero_shot_score.xlsx", index=False)

### Fine-tuned — DPO Scores

In [ ]:
# --- Cloud ---
# df_scored.to_excel("../../outputs/scores/dpo/cloud/dpo_llm_ocr_train_cloud_llama_zero_shot_score.xlsx", index=False)

# --- Device ---
# df_scored.to_excel("../../outputs/scores/dpo/device/dpo_llm_ocr_train_device_llama_zero_shot_score.xlsx", index=False)

## Metric Reference

### QA Evaluation Metric Ranges (Ideal vs Acceptable)

| Metric              | Ideal (Good) | Acceptable Range | Notes |
|---------------------|--------------|------------------|-------|
| **BLEU**            | > 0.6        | 0.3 – 0.6        | Low in QA with short answers; requires exact phrasing |
| **ROUGE-L**         | > 0.5        | 0.3 – 0.5        | Captures longest common subsequence |
| **ROUGE-Lsum**      | > 0.5        | 0.3 – 0.5        | Better suited for longer, sentence-style responses |
| **BERT Precision**  | > 0.85       | 0.75 – 0.85      | Measures semantic precision: how much of prediction is correct |
| **BERT Recall**     | > 0.85       | 0.75 – 0.85      | Measures semantic recall: how much of the reference is captured |
| **BERT F1**         | > 0.85       | 0.75 – 0.85      | Best overall metric for semantic alignment in QA |
| **METEOR**          | > 0.6        | 0.4 – 0.6        | Considers synonyms, stems, and word order |